# Caso Práctico: Streaming + Inferencia en Tiempo Real

Objetivo: simular una arquitectura mínima con ingesta en streaming, **feature store en memoria**, **inferencia ML** online y **sink** de resultados, midiendo latencias y throughput.

---
## 1. Diseño del Flujo
- Ingesta de eventos (simulada)
- Actualización de features por entidad (ventanas cortas)
- Inferencia con un modelo simple (dummy/logístico)
- Persistencia de resultados y métricas operacionales

Esquema:
```

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Observabilidad, Metadata y Linaje para Ecosistemas IA/Big Data (Simulación)](18_observabilidad_metadata_linaje_ia_bigdata_simulacion.ipynb) | Siguiente: [Vector Search y Embeddings (Introducción Práctica)](20_vector_search_embeddings_similitud.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. 1. Diseño del Flujo
2. 2. Loop de Procesamiento en Streaming (simulado)
3. 3. Métricas Operacionales
4. 4. Extensiones Reales

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Caso Práctico: Streaming + Inferencia en Tiempo Real"] --> C["Notebook siguiente"]
```


In [ ]:
import time, random, math
from collections import deque, defaultdict
import pandas as pd
random.seed(13)

class Event:
    def __init__(self, entity_id: str, value: float, ts: float):
        self.entity_id = entity_id
        self.value = value
        self.ts = ts

class EventSource:
    def __init__(self, rate_eps=200, jitter=0.2):
        self.rate = rate_eps
        self.jitter = jitter
    def stream(self, duration_s=2):
        start = time.time()
        emitted = 0
        while time.time() - start < duration_s:
            # Poisson-like arrivals (approx)
            if random.random() < (self.rate*(1+self.jitter*(random.random()-0.5)))/1000.0:
                emitted += 1
                yield Event(entity_id=f'u{random.randint(1,50)}', value=random.random()*10, ts=time.time())
            time.sleep(0.001)

class InMemoryFeatureStore:
    def __init__(self, window_s=5):
        self.window_s = window_s
        self.state = defaultdict(deque)  # entity -> deque of (ts, value)
    def update(self, ev: Event):
        dq = self.state[ev.entity_id]
        dq.append((ev.ts, ev.value))
        # evict old
        cutoff = ev.ts - self.window_s
        while dq and dq[0][0] < cutoff:
            dq.popleft()
    def features(self, entity_id: str):
        dq = self.state[entity_id]
        if not dq:
            return {'count':0,'avg':0.0,'max':0.0}
        vals = [v for _,v in dq]
        return {'count':len(vals), 'avg':sum(vals)/len(vals), 'max':max(vals)}

class DummyModel:
    def __init__(self, w_avg=0.6, w_max=0.4, bias=-3.0):
        self.w_avg, self.w_max, self.bias = w_avg, w_max, bias
    def predict_proba(self, feats: dict):
        z = self.w_avg*feats['avg'] + self.w_max*feats['max'] + self.bias
        p = 1/(1+math.exp(-z))
        return p
    def predict(self, feats: dict, threshold=0.5):
        return int(self.predict_proba(feats) >= threshold)

class ResultsSink:
    def __init__(self):
        self.rows = []
    def add(self, entity_id, ts, proba, pred, latency_ms):
        self.rows.append({'entity_id':entity_id,'ts':ts,'proba':proba,'pred':pred,'latency_ms':latency_ms})
    def df(self):
        return pd.DataFrame(self.rows)

# Wiring
source = EventSource(rate_eps=300)
store = InMemoryFeatureStore(window_s=5)
model = DummyModel(w_avg=0.5, w_max=0.7, bias=-2.2)
results = ResultsSink()


---
## 2. Loop de Procesamiento en Streaming (simulado)
Medimos latencia por evento: (post_inferencia_ts - event.ts)*1000.

In [ ]:
start = time.time()
processed = 0
for ev in source.stream(duration_s=2.5):
    store.update(ev)
    feats = store.features(ev.entity_id)
    proba = model.predict_proba(feats)
    pred = 1 if proba >= 0.6 else 0
    latency_ms = (time.time() - ev.ts) * 1000.0
    results.add(ev.entity_id, ev.ts, proba, pred, latency_ms)
    processed += 1
end = time.time()

elapsed = max(end - start, 1e-9)
print('Eventos procesados:', processed, 'en', f'{elapsed:.2f}s')
print('Throughput aprox:', f'{processed / elapsed:.1f} eventos/s')
results.df().head(5)


---
## 3. Métricas Operacionales
- Throughput (ev/s)
- Latencia P50/P95
- Ratio de alertas (si pred=1)

In [ ]:
df = results.df()
throughput = len(df)/(end-start)
lat_p50 = df['latency_ms'].quantile(0.5)
lat_p95 = df['latency_ms'].quantile(0.95)
alert_rate = (df['pred']==1).mean()

summary = pd.Series({'throughput_eps':throughput,'lat_p50_ms':lat_p50,'lat_p95_ms':lat_p95,'alert_rate':alert_rate})
summary

---
## 4. Extensiones Reales
- Sustituir EventSource por Kafka/Pulsar.
- Feature Store online (Redis/Memcached/RocksDB) y offline (Parquet/Delta).
- Modelo: endpoint de inferencia (REST/gRPC) con batching/caching.
- Router (A/B/Canary/Shadow) y registro de decisiones.
- Persistencia en Data Lake y métricas en Prometheus.

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- Apache Kafka: https://kafka.apache.org/
- NIST AI Risk Management Framework: https://www.nist.gov/itl/ai-risk-management-framework
- Referencias del curso: ../04-recursos/referencias.md
